# SEntFiN Financial Sentiment Classifier
### Entity-Aware Sentiment Analysis using FinBERT

| Field | Value |
|-------|-------|
| Scholar No | 24u021017 |
| Dataset | SEntFiN 1.0 — 10,753 financial news headlines |
| Model | ProsusAI/finbert (fine-tuned) |
| Input | `[CLS] headline [SEP] entity [SEP]` |
| Output | 3-class: positive / negative / neutral |
| Split | 80 / 10 / 10 stratified |

In [ ]:
# ── Cell 1: Download SEntFiN.csv directly from GitHub ──
import os
SENTFIN_CSV = '/content/SEntFiN.csv'
if not os.path.exists(SENTFIN_CSV):
    print('Downloading SEntFiN.csv from GitHub...')
    import subprocess
    subprocess.run(['wget', '-q', '-O', '/content/SEntFiN.csv',
        'https://raw.githubusercontent.com/pyRis/SEntFiN/main/SEntFiN.csv'], check=True)
    print('Download complete!')
else:
    print('SEntFiN.csv already exists.')
DRIVE_PROJECT = '/content'
print(f'File size: {os.path.getsize(SENTFIN_CSV)/1024:.1f} KB')


# ── Cell 2: Confirm dataset ready ──
import os
print('SEntFiN.csv ready:', os.path.exists(SENTFIN_CSV))


In [ ]:
import os, ast, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

BASE_DIR = Path('.')   # outputs land alongside the notebook

CONFIG = {
    "model_name":   "ProsusAI/finbert",
    "max_length":   128,
    "batch_size":   16,
    "epochs":       3,
    "lr":           2e-5,
    "warmup_ratio": 0.10,
    "seed":         42,
    "label2id":     {"negative": 0, "neutral": 1, "positive": 2},
    "id2label":     {0: "negative", 1: "neutral", 2: "positive"},
    "data_path":    SENTFIN_CSV,
    "output_dir":   str(BASE_DIR / "sentfin_finbert"),
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
print(f"Device    : {DEVICE}")
print(f"Model     : {CONFIG['model_name']}")
print(f"Data path : {CONFIG['data_path']}")

## 2. Data Loading & Parsing

In [ ]:
def parse_decisions_robust(s: str) -> dict:
    """Parse Decisions column string to {entity: sentiment} dict.
    Falls back to regex for rows where entity names contain apostrophes
    (e.g. Moody's, Dr Reddy's) that break ast.literal_eval."""
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        pass
    tokens = re.split(r":\s*'(positive|negative|neutral)'", s)
    entities, sentiments = [], []
    for i, tok in enumerate(tokens):
        if tok in ("positive", "negative", "neutral"):
            sentiments.append(tok)
            raw = tokens[i - 1].strip().strip(",").strip().lstrip("{").strip("'")
            entities.append(raw)
    return dict(zip(entities, sentiments))


def load_data(path: str) -> pd.DataFrame:
    """Load SEntFiN.csv and expand to one row per (headline, entity, sentiment) triple."""
    df_raw = pd.read_csv(path)
    records = []
    for _, row in df_raw.iterrows():
        decisions = parse_decisions_robust(str(row["Decisions"]))
        for entity, sentiment in decisions.items():
            sentiment = sentiment.strip().lower()
            if sentiment not in CONFIG["label2id"]:
                continue
            records.append({
                "title":     row["Title"],
                "entity":    entity,
                "sentiment": sentiment,
                "label":     CONFIG["label2id"][sentiment],
            })
    df = pd.DataFrame(records)
    print(f"Expanded rows      : {len(df)}")
    print(f"Label distribution :\n{df['sentiment'].value_counts()}")
    return df

In [ ]:
df = load_data(CONFIG["data_path"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("SEntFiN 1.0 — Dataset EDA", fontweight="bold")

label_counts = df["sentiment"].value_counts()
colors = {"negative": "#e74c3c", "neutral": "#95a5a6", "positive": "#2ecc71"}
axes[0].bar(label_counts.index, label_counts.values,
            color=[colors[l] for l in label_counts.index])
axes[0].set_title("Label Distribution")
axes[0].set_ylabel("Count")
for i, (lbl, cnt) in enumerate(label_counts.items()):
    axes[0].text(i, cnt + 30, str(cnt), ha="center", fontweight="bold")

entity_per_headline = df.groupby("title").size()
axes[1].hist(entity_per_headline.values, bins=range(1, 9), color="#3498db",
             edgecolor="white", align="left")
axes[1].set_title("Entities per Headline")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(BASE_DIR / "sentfin_eda.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Unique headlines : {df['title'].nunique()}")
print(f"Unique entities  : {df['entity'].nunique()}")

## 3. Train / Val / Test Split

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=CONFIG["seed"], stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=CONFIG["seed"], stratify=temp_df["label"]
)

print(f"Train : {len(train_df):>6}  | {dict(train_df['sentiment'].value_counts())}")
print(f"Val   : {len(val_df):>6}  | {dict(val_df['sentiment'].value_counts())}")
print(f"Test  : {len(test_df):>6}  | {dict(test_df['sentiment'].value_counts())}")

## 4. Tokenization & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
print(f"Vocab size: {tokenizer.vocab_size}")

# Verify sentence-pair format: [CLS] headline [SEP] entity [SEP]
sample = tokenizer(
    "SpiceJet to issue 6.4 crore warrants to promoters",
    "SpiceJet",
    max_length=CONFIG["max_length"],
    truncation=True,
    padding="max_length",
    return_tensors="pt",
)
print("Sample decode:", tokenizer.decode(sample["input_ids"][0], skip_special_tokens=False)[:80])

In [ ]:
class SentFinDataset(Dataset):
    """Encodes (headline, entity) pairs as [CLS] headline [SEP] entity [SEP]."""

    def __init__(self, dataframe, tokenizer, max_length):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        enc = self.tokenizer(
            str(row["title"]),
            str(row["entity"]),
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc["token_type_ids"].squeeze(0),
            "label":          torch.tensor(row["label"], dtype=torch.long),
        }

In [ ]:
train_dataset = SentFinDataset(train_df, tokenizer, CONFIG["max_length"])
val_dataset   = SentFinDataset(val_df,   tokenizer, CONFIG["max_length"])
test_dataset  = SentFinDataset(test_df,  tokenizer, CONFIG["max_length"])

# num_workers=0: the SentFinDataset class is defined in __main__, which can't be
# pickled to forkserver workers under nbconvert execution. CPU tokenization is
# fast enough that 0 workers is fine here.
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

batch = next(iter(train_loader))
print(f"input_ids shape: {batch['input_ids'].shape}")

## 5. Model Architecture

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=3,
    id2label=CONFIG["id2label"],
    label2id=CONFIG["label2id"],
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_trainable:,}")
print(model.classifier)

In [ ]:
# Class-weighted loss for mild label imbalance
label_counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor(
    [len(train_df) / (3 * label_counts[i]) for i in range(3)], dtype=torch.float32
).to(DEVICE)
print(f"Class weights: {class_weights.cpu().tolist()}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=0.01, eps=1e-8)

total_steps  = len(train_loader) * CONFIG["epochs"]
warmup_steps = int(CONFIG["warmup_ratio"] * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")

## 6. Training Loop

In [ ]:
def evaluate(model, data_loader, criterion, device):
    """Run inference and return loss, accuracy, macro-F1, preds, labels."""
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels         = batch["label"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
            loss = criterion(outputs.logits, labels)
            total_loss += loss.item()
            all_preds  += torch.argmax(outputs.logits, dim=-1).cpu().tolist()
            all_labels += labels.cpu().tolist()
    avg_loss = total_loss / len(data_loader)
    return avg_loss, accuracy_score(all_labels, all_preds), \
           f1_score(all_labels, all_preds, average="macro"), all_preds, all_labels


def train(model, train_loader, val_loader, optimizer, scheduler,
          criterion, device, epochs, output_dir):
    """Fine-tune model; saves best checkpoint by val macro-F1."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}
    best_val_f1 = 0.0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for step, batch in enumerate(train_loader):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels         = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

            if (step + 1) % 100 == 0:
                print(f"  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} "
                      f"| Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

        avg_train = epoch_loss / len(train_loader)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(avg_train)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        print(f"\nEpoch {epoch+1}/{epochs} | Train Loss: {avg_train:.4f} "
              f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), os.path.join(output_dir, "best_model.pt"))
            print(f"  --> Best model saved (val macro-F1 = {best_val_f1:.4f})")

    return history

In [ ]:
history = train(
    model, train_loader, val_loader,
    optimizer, scheduler, criterion,
    DEVICE, CONFIG["epochs"], CONFIG["output_dir"],
)

## 7. Training Curves

In [ ]:
epochs_range = range(1, CONFIG["epochs"] + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Training History — FinBERT on SEntFiN", fontweight="bold")

axes[0].plot(epochs_range, history["train_loss"], "b-o", label="Train")
axes[0].plot(epochs_range, history["val_loss"],   "r-o", label="Val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(epochs_range, history["val_acc"], "g-o")
axes[1].set_title("Val Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].set_ylim(0, 1)

axes[2].plot(epochs_range, history["val_f1"], "m-o")
axes[2].set_title("Val Macro-F1"); axes[2].set_xlabel("Epoch"); axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(str(BASE_DIR / "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Test Set Evaluation

In [ ]:
import os
ckpt = os.path.join(CONFIG['output_dir'], 'best_model.pt')
if not os.path.exists(ckpt):
    # Fallback: save current model weights as the checkpoint
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    torch.save(model.state_dict(), ckpt)
    print('Warning: no best_model.pt found — using current weights.')

model.load_state_dict(
    torch.load(ckpt, map_location=DEVICE, weights_only=False)
)
print('Loaded best model checkpoint.')

test_loss, test_acc, test_macro_f1, test_preds, test_labels = evaluate(
    model, test_loader, criterion, DEVICE
)

print(f'\n=== Test Set Results ===')
print(f'Loss     : {test_loss:.4f}')
print(f'Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'Macro-F1 : {test_macro_f1:.4f}')
print()
print('=== Classification Report ===')
print(classification_report(
    test_labels, test_preds,
    target_names=['negative', 'neutral', 'positive'],
    digits=4,
))


In [ ]:
cm = confusion_matrix(test_labels, test_preds)
class_names = ["negative", "neutral", "positive"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.set_title("Confusion Matrix — Test Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(str(BASE_DIR / "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

## 9. Save Model & Tokenizer

In [ ]:
import os
os.makedirs(CONFIG['output_dir'], exist_ok=True)
try:
    model.save_pretrained(CONFIG['output_dir'])
    tokenizer.save_pretrained(CONFIG['output_dir'])
    print(f"Saved to: {CONFIG['output_dir']}/")
    for f in sorted(Path(CONFIG['output_dir']).iterdir()):
        print(f'  {f.name}')
except Exception as e:
    print(f'Save warning: {e}')


## 10. Inference Demo

In [ ]:
def predict_sentiment(headline: str, entity: str) -> dict:
    """Predict sentiment for a (headline, entity) pair."""
    model.eval()
    enc = tokenizer(
        headline, entity,
        max_length=CONFIG['max_length'],
        truncation=True, padding='max_length', return_tensors='pt',
    )
    inputs = {
        'input_ids':      enc['input_ids'].to(DEVICE),
        'attention_mask': enc['attention_mask'].to(DEVICE),
    }
    if 'token_type_ids' in enc:
        inputs['token_type_ids'] = enc['token_type_ids'].to(DEVICE)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
    pred  = CONFIG['id2label'][int(torch.argmax(logits, dim=-1).item())]
    return {'label': pred, 'scores': {CONFIG['id2label'][i]: round(probs[i], 4) for i in range(3)}}


examples = [
    ('SpiceJet to issue 6.4 crore warrants to promoters',      'SpiceJet'),
    ('Gold shines on seasonal demand; Silver dull',             'Gold'),
    ('Gold shines on seasonal demand; Silver dull',             'Silver'),
    ('Infosys beats Q4 estimates, raises guidance',             'Infosys'),
    ("Moody's downgrades Tata Steel UK's rating by one notch", 'Tata Steel UK'),
]

print('=== Inference Demo ===\n')
for headline, entity in examples:
    result = predict_sentiment(headline, entity)
    print(f'Headline : {headline}')
    print(f'Entity   : {entity}')
    print(f'Predicted: {result["label"].upper()}  |  Scores: {result["scores"]}')
    print()


---
**Scholar No:** 24u021017 | SEntFiN Financial Sentiment Classifier

**Dataset:** SEntFiN 1.0 — 10,753 headlines expanded to ~14,405 (entity, sentiment) triples  
**Model:** ProsusAI/finbert fine-tuned | Input: `[CLS] headline [SEP] entity [SEP]`  
**Framework:** HuggingFace Transformers + PyTorch | Epochs: 3 | LR: 2e-5 | Batch: 16